# 1. Extracción de datos

Objetivo del libro: estimar con datos colombianos de fuente primaria los insumos de la página
**Costo de capital por sector** de Mirage y evaluar qué modelos los predicen mejor, siguiendo la
metodología del curso aprendizaje automático estándar (validación fuera de muestra y comparación estadística).

| Fuente | Conjunto | Uso |
|---|---|---|
| Superfinanciera, formato 88 (`w9zh-vetq`) | Tasa, monto, CIIU, tamaño, plazo y garantía de cada crédito comercial nuevo, jul-2022 a hoy | P1 · Kd por sector |
| Superfinanciera, Formato 351 (`ur2p-h4yf`) | Portafolio de los fondos de pensiones título a título, 2016 a hoy | P3 · TES, bonos globales |
| Supersociedades NIIF | Balance, resultados y CIIU, ~30.000 empresas por año, 2016–2025 | P2 · estructura de capital |
| Supersociedades, 10.000 más grandes (`6cat-2gcs`) | Empresas de todos los supervisores, 2021–2025 | Representatividad |
| Banco de la República (SDMX) | TES, IBR, tasa de política, TRM, COLCAP | Variables macro |
| FRED | Tesoro 1–30 años, inflación implícita USD, spreads | P3 |
| Damodaran | Betas por industria, primas país | Contraste |

**Configuración.** Con `DESCARGAR = False` se usan los datos ya guardados en `data/processed/`.
Con `True` se vuelven a descargar (la historia completa toma 30–60 minutos).

In [1]:
import sys, warnings, json, time
from pathlib import Path
ML = Path.cwd().parent                      # carpeta ml/
sys.path.insert(0, str(ML))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
PROCESSED, REPORTS = ML / "data" / "processed", ML / "reports"
PRED = REPORTS / "predicciones"; PRED.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", None); pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", palette="deep")

In [2]:
# ===================== CONFIGURACIÓN =====================
DESCARGAR = {
    "fred": False, "banrep": False, "damodaran": False,
    "superfinanciera_tasas": False,      # formato 88, historia completa
    "f351": False,                       # Formato 351
    "supersociedades": False,            # panel NIIF 2016–2025
    "grandes_empresas": False,
}
ANIOS_SUPERSOCIEDADES = (2016, 2025)

In [3]:
from mirage.ingest import fred, banrep, damodaran, superfinanciera, f351, supersociedades, grandes_empresas
t0 = time.time()
if DESCARGAR["fred"]: fred.download()
if DESCARGAR["banrep"]: banrep.download()
if DESCARGAR["damodaran"]: damodaran.industry_betas(); damodaran.country()
if DESCARGAR["superfinanciera_tasas"]: superfinanciera.rates_detail()
if DESCARGAR["f351"]: f351.run()
if DESCARGAR["supersociedades"]: supersociedades.multi_year(range(ANIOS_SUPERSOCIEDADES[0], ANIOS_SUPERSOCIEDADES[1] + 1))
if DESCARGAR["grandes_empresas"]: grandes_empresas.download()
print(f"Descargas: {time.time()-t0:.0f} s")

Descargas: 0 s


## 1.1 Curvas de TES, inflación implícita y spread soberano (a partir del Formato 351)

In [4]:
from mirage.wacc import curves as C
tes = C.add_ytm(pd.read_parquet(PROCESSED / "f351_tes.parquet"))     # tasa desde el precio sucio de valoración
ext = C.add_ytm(pd.read_parquet(PROCESSED / "f351_externa.parquet"))
ust = C.ust_curve(pd.read_parquet(PROCESSED / "fred_series.parquet"), fred.UST_TENORS)
be = C.breakeven(tes); sp, _ = C.sovereign_spread(ext, ust)
be.to_csv(REPORTS / "inflacion_implicita_mensual.csv", index=False); sp.to_csv(REPORTS / "spread_soberano_mensual.csv", index=False)
print(be.tail(3).round(2)); print(sp.tail(3).round(2))

         fecha  tes_cop_2a  tes_uvr_2a  inflacion_implicita_2a  tes_cop_5a  tes_uvr_5a  inflacion_implicita_5a  tes_cop_10a  tes_uvr_10a  inflacion_implicita_10a
119 2026-05-31       14.21        7.19                    6.55       14.05        7.21                    6.38        13.48         6.79                     6.27
120 2026-06-30       12.37        6.29                    5.72       12.06        6.45                    5.27        11.95         6.35                     5.26
121 2026-07-31       12.57        6.09                    6.11       12.30        6.20                    5.74        12.16         6.20                     5.61
         fecha  bonos  col_usd_10a  ust_10a  spread_soberano_10a_pp  spread_mediano_bonos_pp  rmse_ns
124 2026-05-31     33         7.01     4.45                    2.56                     2.50     0.14
125 2026-06-30     33         6.54     4.44                    2.10                     2.01     0.12
126 2026-07-31     31         6.85     4.75   

## 1.2 Inventario

In [5]:
inv = []
for p in sorted(PROCESSED.glob("*.parquet")):
    d = pd.read_parquet(p); fc = next((c for c in ("fecha", "mes", "anio") if c in d.columns), None)
    inv.append({"conjunto": p.stem, "filas": len(d), "columnas": d.shape[1],
                "desde": str(d[fc].min())[:10] if fc else "", "hasta": str(d[fc].max())[:10] if fc else ""})
pd.DataFrame(inv)

,conjunto,filas,columnas,desde,hasta
0,banrep_series,61333,8,1967-03-22,2026-10-08
1,betas_no_cotizadas,24842,13,,
2,damodaran_betas_emerg,94,13,,
3,dataset_p1,691,25,2022-07-01,2026-06-01
4,dataset_p2,65739,29,2016,2024
5,dataset_p3,127,11,,
6,f351_acciones,5009,19,2016-01-31,2026-07-31
7,f351_bonos,31767,19,2016-01-31,2026-07-31
8,f351_externa,3144,19,2016-01-31,2026-07-31
9,f351_tes,2714,19,2016-01-31,2026-07-31
